In [ ]:
from importlib import reload
import plotting_utils as pu

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

freq = 32

cell_types = ["Exc", "Pvalb", "Sst", "Vip", "Htr3a"]
colors = [
    "tab:red",
    "tab:blue",
    "tab:green",
    "tab:purple",
    "tab:purple",
]
layers = ["L1", "L2/3", "L4", "L5", "L6"]

cell_labels = {
    "Exc": "Exc",
    "Pvalb": "Pvalb",
    "Sst": "Sst",
    "Vip": "Vip",
    "Htr3a": "L1 Inh",
}

In [ ]:
net_name = "core"  # change to "core_<variant>" for a model variant
pu.plot_raster_movie(
    config_file=f"{net_name}/configs/8dir_10trials_-2000_silencing_Pvalb_100perc_checkpoint/config_0.json",
    freq=freq,
    cell_types=cell_types,
    cell_labels=cell_labels,
    color_order=colors,
    bins=np.linspace(0, 3001, 301),
    n_xticks=6,
    ratio=[1, 20],
    s=0.3,
    legend=True,
    figsize=(5, 5),
    save_path="paper_figs/fig3A_dg_raster_Pvalb_silencing_Jul17_2026.tiff",
)
# fig.savefig("raster_drifting_gratings_alldirs.pdf", dpi=600, bbox_inches="tight")

In [ ]:
pu.plot_raster_movie(
    config_file=f"{net_name}/configs/8dir_10trials_-2000_silencing_Sst_100perc_checkpoint/config_0.json",
    freq=freq,
    cell_types=cell_types,
    cell_labels=cell_labels,
    color_order=colors,
    bins=np.linspace(0, 3001, 301),
    n_xticks=6,
    ratio=[1, 20],
    s=0.3,
    legend=True,
    figsize=(5, 5),
    save_path="paper_figs/fig3_dg_raster_Sst_silencing_Jul17_2026.tiff",
)
# fig.savefig("raster_drifting_gratings_alldirs.pdf", dpi=600, bbox_inches="tight")

## Fig 3 suppl: firing rates distribution for inhibitory population silencing

In [ ]:
Perturb_dict = {
    "no perturb": "_checkpoint",
    "silencing Vip": "_-2000_silencing_Vip_100perc_checkpoint",
    "silencing Pvalb": "_-2000_silencing_Pvalb_100perc_checkpoint",
    "silencing Sst": "_-2000_silencing_Sst_100perc_checkpoint",
    "silencing i23Vip": "_-2000_silencing_i23Vip_100perc_checkpoint",
    "silencing i23Pvalb": "_-2000_silencing_i23Pvalb_100perc_checkpoint",
    "silencing i23Sst": "_-2000_silencing_i23Sst_100perc_checkpoint",
    "silencing i4Vip": "_-2000_silencing_i4Vip_100perc_checkpoint",
    "silencing i4Pvalb": "_-2000_silencing_i4Pvalb_100perc_checkpoint",
    "silencing i4Sst": "_-2000_silencing_i4Sst_100perc_checkpoint",
    "silencing i5Vip": "_-2000_silencing_i5Vip_100perc_checkpoint",
    "silencing i5Pvalb": "_-2000_silencing_i5Pvalb_100perc_checkpoint",
    "silencing i5Sst": "_-2000_silencing_i5Sst_100perc_checkpoint",
    "silencing i6Vip": "_-2000_silencing_i6Vip_100perc_checkpoint",
    "silencing i6Pvalb": "_-2000_silencing_i6Pvalb_100perc_checkpoint",
    "silencing i6Sst": "_-2000_silencing_i6Sst_100perc_checkpoint",
}

config_files_path = f"{net_name}/configs"


def get_condition_color(condition):
    if condition == "no perturb":
        return "white"
    elif "Vip" in condition:
        return "tab:purple"
    elif "Pvalb" in condition:
        return "tab:blue"
    elif "Sst" in condition:
        return "tab:green"
    return "gray"


perturb_palette = {cond: get_condition_color(cond) for cond in Perturb_dict}

In [ ]:
v1df_all = pd.DataFrame()
for condition, file_info in Perturb_dict.items():
    print(condition)
    file_name = (
        config_files_path
        + f"/8dir_10trials{file_info}"
        + "/config_0.json"
    )
    v1df, spike_df = pu.data_organize(
        file_name, t_min=500, t_max=3000, radius=pu.settings["core"]["radius"]
    )
    v1df["condition"] = condition
    v1df_all = pd.concat([v1df_all, v1df])

v1df_all["cell_type"] = v1df_all["cell_type"].map(cell_labels)

v1df_all["cell_layer"] = v1df_all.apply(
    lambda row: row["layer"] + " " + row["cell_type"] if row["layer"] != "L1" else "L1 Inh", axis=1
)

layer_cell = v1df_all["cell_layer"].unique().tolist()

fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
fig.patch.set_facecolor("none")
sns.boxplot(
    ax=ax,
    data=v1df_all,
    x="cell_layer",
    y="fr_stimulus",
    hue="condition",
    palette=perturb_palette,
    flierprops=dict(marker="x", markeredgecolor="black", markersize=2, zorder=2),
    medianprops=dict(color="tab:brown", alpha=0.7, zorder=2),
    boxprops=dict(linewidth=0.5, zorder=2),
    capprops=dict(color="black", linewidth=0.5, zorder=2),
    order=layer_cell,
)
# plt.ylim([-1, 400])
plt.legend(fontsize="x-small")  # , bbox_to_anchor=(1.1, 1), loc='upper center', ncol=1)
plt.xticks(rotation=45)
plt.xlabel(None)
plt.ylabel("Firing rate [spk/s]")

# plot background layer divider
layer_sep = np.cumsum(v1df_all.groupby("layer")["cell_type"].nunique().values) - 0.5
for i in range(1, len(layer_sep), 2):
    ax.axvspan(layer_sep[i - 1], layer_sep[i], color="gray", alpha=0.1)
# plot layer text
xlim = ax.get_xlim()
ylim = ax.get_ylim()
layer_sep = np.insert(layer_sep, 0, xlim[0])
layer_sep[-1] = xlim[1]
layers = ["L1", "L2/3", "L4", "L5", "L6"]
for i in range(1, len(layer_sep)):
    plt.text(
        (layer_sep[i - 1] + layer_sep[i]) / 2 - 0.3,
        ylim[1] * 0.8,
        layers[i - 1],
        fontsize=12,
        weight="bold",
    )

In [ ]:
subplot_conditions = [
    ["no perturb", "silencing Vip", "silencing Sst", "silencing Pvalb"],
    ["no perturb", "silencing i23Pvalb", "silencing i4Pvalb", "silencing i5Pvalb", "silencing i6Pvalb"],
    ["no perturb", "silencing i23Sst", "silencing i4Sst", "silencing i5Sst", "silencing i6Sst"],
    ["no perturb", "silencing i23Vip", "silencing i4Vip", "silencing i5Vip", "silencing i6Vip"],
]
subplot_titles = [
    r"$\mathbf{A.}$ Whole pop. silencing",
    r"$\mathbf{B.}$ Pvalb silencing",
    r"$\mathbf{C.}$ Sst silencing",
    r"$\mathbf{D.}$ Vip silencing",
]
subplot_legend_labels = [
    None,  # whole-pop: use default labels
    ['no perturb', 'silencing L2/3 Pvalb', 'silencing L4 Pvalb', 'silencing L5 Pvalb', 'silencing L6 Pvalb'],
    ['no perturb', 'silencing L2/3 Sst', 'silencing L4 Sst', 'silencing L5 Sst', 'silencing L6 Sst'],
    ['no perturb', 'silencing L2/3 Vip', 'silencing L4 Vip', 'silencing L5 Vip', 'silencing L6 Vip'],
]

# For the layer-specific subplots, generate 4 shades per cell type
pvalb_shades = plt.colormaps["Blues"](np.linspace(0.35, 0.9, 4)).tolist()
sst_shades   = plt.colormaps["Greens"](np.linspace(0.35, 0.9, 4)).tolist()
vip_shades   = plt.colormaps["Purples"](np.linspace(0.35, 0.9, 4)).tolist()

subplot_palettes = [
    perturb_palette,  # whole-pop: use the standard palette
    {"no perturb": "white", "silencing i23Pvalb": pvalb_shades[0],
     "silencing i4Pvalb": pvalb_shades[1], "silencing i5Pvalb": pvalb_shades[2],
     "silencing i6Pvalb": pvalb_shades[3]},
    {"no perturb": "white", "silencing i23Sst": sst_shades[0],
     "silencing i4Sst": sst_shades[1], "silencing i5Sst": sst_shades[2],
     "silencing i6Sst": sst_shades[3]},
    {"no perturb": "white", "silencing i23Vip": vip_shades[0],
     "silencing i4Vip": vip_shades[1], "silencing i5Vip": vip_shades[2],
     "silencing i6Vip": vip_shades[3]},
]



In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(8, 10), dpi=300)
fig.patch.set_facecolor("none")

for i, (ax, conditions, title, sub_palette, legend_labels) in enumerate(zip(
    axes, subplot_conditions, subplot_titles, subplot_palettes, subplot_legend_labels
)):
    sub_df = v1df_all[v1df_all["condition"].isin(conditions)]
    sns.boxplot(
        ax=ax,
        data=sub_df,
        x="cell_layer",
        y="fr_stimulus",
        hue="condition",
        hue_order=conditions,
        palette=sub_palette,
        flierprops=dict(marker="x", markeredgecolor="black", markersize=2, zorder=2),
        medianprops=dict(color="tab:brown", alpha=0.7, zorder=2),
        boxprops=dict(linewidth=0.5, zorder=2),
        capprops=dict(color="black", linewidth=0.5, zorder=2),
        order=layer_cell,
    )
    ax.set_title(title, fontsize=10)
    if legend_labels is not None:
        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=legend_labels, fontsize="x-small")
    else:
        ax.legend(fontsize="x-small")
    ax.set_xlabel(None)
    ax.set_ylabel("Firing rate [spk/s]")
    if i == len(axes) - 1:
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    else:
        ax.tick_params(labelbottom=False)

    # plot background layer divider
    layer_sep = np.cumsum(sub_df.groupby("layer")["cell_type"].nunique().values) - 0.5
    for j in range(1, len(layer_sep), 2):
        ax.axvspan(layer_sep[j - 1], layer_sep[j], color="gray", alpha=0.1)
    # plot layer text
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    layer_sep_ext = np.insert(layer_sep, 0, xlim[0])
    layer_sep_ext[-1] = xlim[1]
    layers = ["L1", "L2/3", "L4", "L5", "L6"]
    for j in range(1, len(layer_sep_ext)):
        ax.text(
            (layer_sep_ext[j - 1] + layer_sep_ext[j]) / 2 - 0.3,
            ylim[1] * 0.8,
            layers[j - 1],
            fontsize=12,
            weight="bold",
        )

fig.tight_layout()
fig.savefig(
    "paper_figs/fig3_suppl_dg_firing_rate_dist_inh_pop_sil_Jul17_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(10, 10), dpi=300, sharey=True)
fig.patch.set_facecolor("none")

for i, (ax, conditions, title, sub_palette, legend_labels) in enumerate(zip(
    axes, subplot_conditions, subplot_titles, subplot_palettes, subplot_legend_labels
)):
    sub_df = v1df_all[v1df_all["condition"].isin(conditions)]
    sns.boxplot(
        ax=ax,
        data=sub_df,
        x="fr_stimulus",
        y="cell_layer",
        hue="condition",
        hue_order=conditions,
        palette=sub_palette,
        flierprops=dict(marker="x", markeredgecolor="black", markersize=2, zorder=2),
        medianprops=dict(color="tab:brown", alpha=0.7, zorder=2),
        boxprops=dict(linewidth=0.5, zorder=2),
        capprops=dict(color="black", linewidth=0.5, zorder=2),
        order=layer_cell,
    )
    ax.set_title(title, fontsize=10)
    if legend_labels is not None:
        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=legend_labels, fontsize="x-small")
    else:
        ax.legend(fontsize="x-small")
    ax.set_xlabel("Firing rate [spk/s]", fontsize=10)
    if i == 0:
        ax.set_ylabel("Neuron population")
    else:
        ax.set_ylabel(None)
        ax.tick_params(labelleft=False)

    # horizontal layer background bands
    layer_sep = np.cumsum(sub_df.groupby("layer")["cell_type"].nunique().values) - 0.5
    for j in range(1, len(layer_sep), 2):
        ax.axhspan(layer_sep[j - 1], layer_sep[j], color="gray", alpha=0.1)

    # layer text on rightmost subplot only
    if i == len(axes) - 1:
        ylim = ax.get_ylim()
        # seaborn inverts the y-axis for horizontal categorical plots, so
        # ylim[1] is the near end (-0.5) and ylim[0] is the far end (n-0.5).
        # Use min(ylim) as the start; cumsum already gives the correct far boundary.
        layer_sep_ext = np.insert(layer_sep, 0, min(ylim))
        layer_names = ["L1", "L2/3", "L4", "L5", "L6"]
        for j in range(1, len(layer_sep_ext)):
            ax.text(
                1.03,
                (layer_sep_ext[j - 1] + layer_sep_ext[j]) / 2,
                layer_names[j - 1],
                transform=ax.get_yaxis_transform(),
                fontsize=10,
                weight="bold",
                va="center",
            )

fig.tight_layout()
fig.savefig(
    "paper_figs/fig3_suppl_dg_fr_dist_inh_pop_sil_Jul27_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)

### Condensed heatmap version of Fig. S3 A–D (reviewer response)

One small **anatomical heatmap per perturbation**, arranged in the same four rows
as the original figure (whole-population, then layer-specific Pvalb / Sst / Vip).
Within each subplot the **y-axis is layer** and the **x-axis is cell type**
(Exc, Pvalb, Sst, Vip; L1 Htr3a is shown in the Vip column), and **pixel color is
the mean stimulus-evoked firing rate normalized to the no-perturbation
condition** (fold change). The subplot title is the perturbation label, bold
outlines flag ≥ 2-fold changes, and the directly silenced target population is
hatched.

Saved to `paper_figs/fig3_suppl_dg_firing_rate_heatmap_norm_inh_pop_sil_July16_2026.pdf`.

In [ ]:
# ===========================================================================
# Per-perturbation anatomical heatmaps (reviewer response to Fig. S3 A-D).
# One small heatmap per silencing condition, arranged in the same 4 rows as
# the original figure.  Within each subplot: y = layer, x = cell type
# (Exc/Pvalb/Sst/Vip, with L1 Htr3a shown in the Vip column); color = mean
# firing rate NORMALIZED to the no-perturbation condition (fold change).
# Requires `v1df_all` from the cell above.
# ===========================================================================
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm

row_layers = ["L1", "L2/3", "L4", "L5", "L6"]     # subplot y axis (top -> bottom)
col_cells = ["Exc", "Pvalb", "Sst", "Vip"]        # subplot x axis; L1 Htr3a -> Vip

# the 4 rows of the original figure and the per-condition subplot titles
panel_rows = [
    ("Whole population silencing", [
        ("silencing Vip", "silencing Vip"),
        ("silencing Sst", "silencing Sst"),
        ("silencing Pvalb", "silencing Pvalb"),
    ]),
    ("Layer-specific Pvalb silencing", [
        ("silencing i23Pvalb", "silencing L2/3 Pvalb"),
        ("silencing i4Pvalb", "silencing L4 Pvalb"),
        ("silencing i5Pvalb", "silencing L5 Pvalb"),
        ("silencing i6Pvalb", "silencing L6 Pvalb"),
    ]),
    ("Layer-specific Sst silencing", [
        ("silencing i23Sst", "silencing L2/3 Sst"),
        ("silencing i4Sst", "silencing L4 Sst"),
        ("silencing i5Sst", "silencing L5 Sst"),
        ("silencing i6Sst", "silencing L6 Sst"),
    ]),
    ("Layer-specific Vip silencing", [
        ("silencing i23Vip", "silencing L2/3 Vip"),
        ("silencing i4Vip", "silencing L4 Vip"),
        ("silencing i5Vip", "silencing L5 Vip"),
        ("silencing i6Vip", "silencing L6 Vip"),
    ]),
]
ncols = max(len(conds) for _, conds in panel_rows)

# mean firing rate per (population, condition) and the no-perturb reference
mean_fr = v1df_all.groupby(["cell_layer", "condition"])["fr_stimulus"].mean().unstack()
baseline_pop = mean_fr["no perturb"]


def pop_name(layer, cell):
    """cell_layer key for a (layer, x-column) slot; L1 Vip column holds Htr3a."""
    if layer == "L1":
        return "L1 Htr3a" if cell == "Vip" else None
    return f"{layer} {cell}"


layer_of = {"i23": "L2/3", "i4": "L4", "i5": "L5", "i6": "L6"}
def targets(cond):
    key = cond.replace("silencing ", "")
    for pre, lay in layer_of.items():
        if key.startswith(pre):
            return {(lay, key[len(pre):])}     # layer-specific: one population
    return {(l, key) for l in row_layers}      # whole-pop: cell type, all layers


def matrices(cond):
    """(fold-change grid, target-mask grid) as len(row_layers) x len(col_cells)."""
    fc = np.full((len(row_layers), len(col_cells)), np.nan)
    tg = np.zeros_like(fc, dtype=bool)
    tset = targets(cond)
    for i, lay in enumerate(row_layers):
        for j, cell in enumerate(col_cells):
            p = pop_name(lay, cell)
            if p is None or p not in mean_fr.index:
                continue
            b = baseline_pop.get(p, np.nan)
            fc[i, j] = mean_fr.loc[p, cond] / b if b else np.nan
            ct = "Htr3a" if p.endswith("Htr3a") else cell
            tg[i, j] = (lay, ct) in tset
    return fc, tg


LIM = 2.0                                       # +/- 2 -> 4x change saturates
NORM = TwoSlopeNorm(vmin=-LIM, vcenter=0.0, vmax=LIM)
CMAP = plt.get_cmap("RdBu_r").copy()
CMAP.set_bad("0.9")                             # absent populations -> light gray


def draw_subplot(ax, cond, title, show_y, show_x):
    fc, tg = matrices(cond)
    log2 = np.where(fc == 0, -np.inf, np.log2(np.where(fc == 0, np.nan, fc)))
    disp = np.ma.masked_invalid(
        np.where(np.isnan(fc), np.nan, np.clip(log2, -LIM, LIM)))
    ax.imshow(disp, cmap=CMAP, norm=NORM, aspect="equal")
    for i in range(len(row_layers)):
        for j in range(len(col_cells)):
            if np.isnan(fc[i, j]):
                continue
            v = fc[i, j]
            txt = "0" if v == 0 else (f"{v:.1f}×" if v < 10 else f"{v:.0f}×")
            mag = 1.0 if v == 0 else min(abs(np.log2(v)) / LIM, 1.0)
            ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                    color="white" if mag > 0.55 else "black")
            if tg[i, j]:                        # directly silenced -> hatch
                ax.add_patch(Rectangle((j - .5, i - .5), 1, 1, fill=False,
                             hatch="////", edgecolor="0.25", lw=0.0))
            elif v != 0 and abs(np.log2(v)) >= 1.0:    # >=2-fold network effect
                ax.add_patch(Rectangle((j - .5, i - .5), 1, 1, fill=False,
                             edgecolor="black", lw=1.8))
    ax.set_xticks(range(len(col_cells)))
    ax.set_yticks(range(len(row_layers)))
    ax.set_xticklabels(col_cells if show_x else [], fontsize=8, rotation=45,
                       ha="right")
    ax.set_yticklabels(row_layers if show_y else [], fontsize=8)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, fontsize=9.5)


fig = plt.figure(figsize=(11.5, 12.8), dpi=300)
outer = fig.add_gridspec(len(panel_rows), 1, hspace=0.55)
for ri, (group_title, conds) in enumerate(panel_rows):
    inner = outer[ri].subgridspec(1, ncols, wspace=0.15)
    for ci, (cond, title) in enumerate(conds):
        ax = fig.add_subplot(inner[0, ci])
        draw_subplot(ax, cond, title, show_y=(ci == 0), show_x=True)
    row_ax = fig.add_subplot(outer[ri])         # transparent row for its label
    row_ax.axis("off")
    row_ax.text(-0.045, 0.5, group_title, rotation=90, ha="center", va="center",
                fontsize=11, fontweight="bold", transform=row_ax.transAxes)

# shared colorbar
sm = plt.cm.ScalarMappable(norm=NORM, cmap=CMAP)
cax = fig.add_axes([0.93, 0.30, 0.015, 0.4])
cbar = fig.colorbar(sm, cax=cax, ticks=[-2, -1, 0, 1, 2])
cbar.ax.set_yticklabels(["0.25×", "0.5×", "1×", "2×", "4×"])
cbar.set_label("Firing rate relative to no perturbation", fontsize=10)

fig.suptitle("Effect of inhibitory silencing on firing rate "
             "(normalized to no perturbation)",
             fontsize=13, fontweight="bold", x=0.5, y=0.995)
fig.text(0.5, 0.055,
         "Color = mean firing rate ÷ no-perturbation rate.   "
         "Bold outline = |change| ≥ 2-fold.   Hatched = directly silenced "
         "target.   Gray = population absent.   L1 Vip column = L1 Htr3a.",
         ha="center", fontsize=8.5, style="italic")
fig.subplots_adjust(left=0.07, right=0.91, top=0.94, bottom=0.09)
fig.savefig(
    "paper_figs/fig3_suppl_dg_firing_rate_heatmap_norm_inh_pop_sil_July16_2026.pdf",
    dpi=300, bbox_inches="tight", facecolor="white",
)